# Project Notebook: BERT Encoder Sentiment Classification on IMDB

In this project you will fine-tune a BERT-style encoder model to classify IMDB movie reviews as **negative** or **positive**.

We use `distilbert/distilbert-base-uncased` because it teaches the same encoder workflow as BERT but trains faster for guided project use.

**What you will build:** a complete text-classification pipeline.

**Why it matters:** most real NLP projects follow this same pattern: load text, tokenize it, fine-tune a pretrained model, evaluate predictions, and inspect mistakes.

**Example flow:**

```text
movie review text
-> tokenizer
-> input_ids + attention_mask
-> BERT-style encoder
-> sequence classification head
-> logits [batch_size, 2]
-> NEGATIVE or POSITIVE
```

**Note:** if you want full BERT later, change only this checkpoint in the configuration cell:

```python
MODEL_CHECKPOINT = "google-bert/bert-base-uncased"
```


## 1. Project Goals

By the end of this notebook, you should be able to:

1. Explain why BERT and DistilBERT are **encoder-style** models.
2. Load the IMDB sentiment dataset and inspect its text and labels.
3. Convert raw reviews into `input_ids`, `attention_mask`, and `labels`.
4. Explain why dynamic padding is better than padding every review to one fixed length.
5. Fine-tune `AutoModelForSequenceClassification` on a balanced project subset.
6. Read accuracy, precision, recall, and F1 without treating them as magic numbers.
7. Run inference on fresh reviews and explain why the prediction makes sense.
8. Complete a short project report using evidence from the notebook outputs.


## Project Deliverables

By the end, you should have:

- a runnable notebook with clean outputs,
- a baseline metric summary,
- one small experiment that changes a configuration value,
- at least three inspected predictions or mistakes,
- a short written report using the template at the end.

**What to submit:** this completed notebook and your report answers.

**Why this format:** it mirrors a real ML project: build, measure, inspect, explain, and improve.


## 2. Setup Cell

Run the install line only in a fresh environment such as Google Colab.

**What:** installs Hugging Face, Datasets, Accelerate, and PyTorch.

**Why:** these libraries provide the tokenizer, pretrained encoder, dataset loader, and training loop.

**Example:** if `import transformers` fails later, come back here, uncomment the install line, run it, then restart the notebook runtime.

**Note:** keep the line commented when the packages are already installed. Reinstalling packages in the middle of a notebook can make debugging harder.


In [1]:
# Uncomment and run this line only in a fresh environment, such as Google Colab.
# %pip install -q "transformers>=4.40" "datasets>=2.18" "accelerate>=1.1.0" "torch"


## 3. Imports

**What:** import the tools we will use for data loading, tokenization, modeling, metrics, and training.

**Why:** keeping imports in one early cell makes the notebook reproducible from a clean restart.

**Example:** `AutoTokenizer` loads the correct tokenizer for the checkpoint, while `AutoModelForSequenceClassification` loads the encoder plus a classifier head.

**Note:** if an import fails, fix the setup cell first instead of changing later code cells.


In [2]:
# Standard-library imports help with inspection, reproducibility, label counting, and file paths.
import inspect
import random
from collections import Counter
from pathlib import Path

# NumPy and PyTorch are the numeric backbones used for metrics and tensors.
import numpy as np
import torch
# Datasets gives us the IMDB corpus and dataset container classes.
from datasets import Dataset, DatasetDict, load_dataset
# These Hugging Face classes cover tokenization, model loading, padding, training, and inference.
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)

# Import the package itself so we can print the installed version for debugging and compatibility.
import transformers

print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)

Transformers version: 5.9.0
Torch version: 2.10.0


## 4. Configuration

This is the main control panel for the project.

**What:** choose the model, dataset, sequence length, subset sizes, training hyperparameters, and output folder.

**Why:** you can run a small experiment by changing a few clear variables instead of searching through the whole notebook.

**Example:** lowering `MAX_LENGTH` usually trains faster, but may remove useful words from long reviews.

**Note:** for a very quick CPU smoke test, set `MAX_TRAIN_EXAMPLES = 200`, `MAX_EVAL_EXAMPLES = 80`, and `RUN_TRAINING = False` first. Then turn training back on when the pipeline is understood.


In [3]:
# The checkpoint controls both the tokenizer family and the pretrained encoder weights.
MODEL_CHECKPOINT = "distilbert/distilbert-base-uncased"

# Dataset name used by Hugging Face Datasets.
DATASET_NAME = "stanfordnlp/imdb"

# A fixed seed makes shuffling and weight initialization more reproducible.
SEED = 42

# Project experiment knob: shorter reviews train faster, but may lose information.
MAX_LENGTH = 256

# Project experiment knob: small subsets keep the project runnable during a guided run.
MAX_TRAIN_EXAMPLES = 1200
MAX_EVAL_EXAMPLES = 300

# Fine-tuning defaults for a pretrained encoder classifier.
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01

# Set to False when you only want to inspect preprocessing, shapes, and code flow.
RUN_TRAINING = True

# Keep project outputs separate from the detailed reference notebook outputs.
OUTPUT_DIR = Path("outputs/bert_imdb_sentiment_project_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_CHECKPOINT)
print("Dataset:", DATASET_NAME)
print("Output directory:", OUTPUT_DIR)


Model: distilbert/distilbert-base-uncased
Dataset: stanfordnlp/imdb
Output directory: outputs/bert_imdb_sentiment_project_notebook


## 5. Reproducibility And Device

Random seeds make the guided run more stable.

`device` tells us where tensors will live:

- `cuda`: NVIDIA GPU.
- `mps`: Apple Silicon GPU.
- `cpu`: normal processor.

The model and every tensor in a forward pass must be on the same device.

In [4]:
def set_seed(seed: int) -> None:
    # Python's built-in random module is used by some helper utilities and shuffling paths.
    random.seed(seed)
    # NumPy randomness can affect metrics or any auxiliary numeric work.
    np.random.seed(seed)
    # Torch seed controls weight initialization and many data-order operations.
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        # If multiple CUDA devices exist, seed all of them.
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

# Prefer CUDA when available, then Apple Silicon MPS, and fall back to CPU otherwise.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


## 6. Load the IMDB Dataset

The IMDB dataset contains movie reviews and a binary sentiment label.

```text
0 -> negative
1 -> positive
```

**What:** load the IMDB dataset directly from Hugging Face.

**Why:** the rest of the notebook uses these real review texts and labels for tokenization, training, and evaluation.

**Example:** one training row looks like `{"text": "...review...", "label": 1}`.

**Note:** this cell needs dataset access through Hugging Face. If it fails, check the package installation and network/cache setup before continuing.


In [5]:
# Load the real IMDB dataset once so every later cell uses the same data object.
raw_datasets = load_dataset(DATASET_NAME)
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

## 7. Inspect the Raw Data

Before modeling, always inspect the dataset.

**What:** print the dataset splits, column names, feature types, labels, and example reviews.

**Why:** many machine learning bugs come from assuming the wrong column name or label format.

**Example:** this notebook expects a `text` column and a numeric `label` column.

**Note:** do this inspection before writing preprocessing code in any NLP project.


In [6]:
print(raw_datasets)
print("\nTrain columns:", raw_datasets["train"].column_names)
print("Train features:", raw_datasets["train"].features)

for i in range(2):
    row = raw_datasets["train"][i]
    print("\nExample", i)
    print("label:", row["label"])
    print("text preview:", row["text"][:400].replace("\n", " "))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Train columns: ['text', 'label']
Train features: {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

Example 0
label: 0
text preview: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student name

Example 1
label: 0
text preview: "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are becaus

### Checkpoint 1: Data Understanding

Pause and answer before moving on:

1. What are the dataset split names?
2. Which column contains the review text?
3. Which column contains the sentiment label?
4. Are labels stored as words or integers?

**Why this matters:** if you cannot describe the raw data, later model results will be hard to trust.


## 8. Label Names

Neural networks work with numeric labels, but humans need readable names.

**What:** create both directions of the mapping.

```text
id2label: 0 -> NEGATIVE, 1 -> POSITIVE
label2id: NEGATIVE -> 0, POSITIVE -> 1
```

**Why:** storing these mappings in the model config makes inference outputs easier to read.

**Example:** the pipeline can return `POSITIVE` instead of just `1`.

**Note:** label mappings are part of the model contract. Keep them consistent from training to inference.


In [7]:
# Human-readable names are nicer in metrics tables and pipeline outputs than bare integers.
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

print("id2label:", id2label)
print("label2id:", label2id)
print("Train label counts:", Counter(raw_datasets["train"]["label"]))
print("Test label counts:", Counter(raw_datasets["test"]["label"]))

id2label: {0: 'NEGATIVE', 1: 'POSITIVE'}
label2id: {'NEGATIVE': 0, 'POSITIVE': 1}
Train label counts: Counter({0: 12500, 1: 12500})
Test label counts: Counter({0: 12500, 1: 12500})


## 9. Make a Balanced Classroom Subset

Full IMDB training is useful, but slower than we need for a guided project.

**What:** make smaller train/eval splits while keeping both labels represented.

**Why:** a balanced subset avoids the model seeing mostly one class and keeps runtime reasonable.

**Example:** if `MAX_TRAIN_EXAMPLES = 1200`, the helper tries to select about 600 negative and 600 positive reviews.

**Note:** increasing the subset size usually improves learning but costs more time.


In [8]:
def balanced_subset(split: Dataset, max_examples: int, seed: int) -> Dataset:
    # Shuffle first so we do not accidentally take a biased slice from the start of the dataset.
    shuffled = split.shuffle(seed=seed)
    labels = list(shuffled["label"])
    # For binary sentiment, half of max_examples per class keeps the subset balanced.
    per_label = max_examples // 2

    selected_indices = []
    counts = Counter()

    for idx, label in enumerate(labels):
        # Keep taking rows for a class until that class reaches its quota.
        if counts[label] < per_label:
            selected_indices.append(idx)
            counts[label] += 1
        if len(selected_indices) >= max_examples:
            break

    # `select` keeps only the chosen rows while preserving Hugging Face Dataset behavior.
    return shuffled.select(selected_indices)


# Build the project-sized splits used by training and evaluation.
train_subset = balanced_subset(raw_datasets["train"], MAX_TRAIN_EXAMPLES, SEED)
eval_subset = balanced_subset(raw_datasets["test"], MAX_EVAL_EXAMPLES, SEED)

# We rename the held-out split to `eval` because that name reads cleanly later in the notebook.
small_datasets = DatasetDict(train=train_subset, eval=eval_subset)

print(small_datasets)
print("Small train counts:", Counter(small_datasets["train"]["label"]))
print("Small eval counts:", Counter(small_datasets["eval"]["label"]))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1200
    })
    eval: Dataset({
        features: ['text', 'label'],
        num_rows: 300
    })
})
Small train counts: Counter({1: 600, 0: 600})
Small eval counts: Counter({1: 150, 0: 150})


## 10. Load The Tokenizer

`AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)` reads the model ID and loads the matching tokenizer.

For DistilBERT/BERT this is a WordPiece-style tokenizer.

Important tokenizer concepts:

- `vocab_size`: number of known token IDs.
- `pad_token`: token used to fill shorter sequences in a batch.
- `cls_token`: classification-start token.
- `sep_token`: separator/end token.
- `model_max_length`: maximum length supported by the tokenizer/model family.

In [9]:
# AutoTokenizer picks the correct tokenizer class based on the checkpoint string.
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

print("Tokenizer class:", type(tokenizer).__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("CLS token:", tokenizer.cls_token, tokenizer.cls_token_id)
print("SEP token:", tokenizer.sep_token, tokenizer.sep_token_id)
print("Tokenizer model_max_length:", tokenizer.model_max_length)

Tokenizer class: BertTokenizer
Vocab size: 30522
Pad token: [PAD] 0
CLS token: [CLS] 101
SEP token: [SEP] 102
Tokenizer model_max_length: 512


## 11. Tokenize One Example by Hand

This is the best place to slow down and look closely.

**What:** convert one raw review into model-ready tensors.

**Why:** tokenization is where human language becomes numbers the model can process.

**Example:** `"unbelievable"` may become one token or multiple subword tokens depending on the tokenizer vocabulary.

**Note:** here we use `padding="max_length"` only to make one example easy to inspect. During training we will use dynamic padding.


In [10]:
# Grab one concrete example so you can inspect tokenization on a real review.
example_text = small_datasets["train"][0]["text"]
example_label = small_datasets["train"][0]["label"]

# The tokenizer converts raw text into integer IDs plus an attention mask.
encoded_example = tokenizer(
    example_text,
    max_length=MAX_LENGTH,
    truncation=True,
    padding="max_length",
    # Returning PyTorch tensors lets us send this output directly into the model.
    return_tensors="pt",
)

print("Original label:", example_label, "->", id2label[example_label])
print("Original text preview:", example_text[:350].replace("\n", " "))
print("\nTokenizer output keys:", encoded_example.keys())

for name, tensor in encoded_example.items():
    print(f"{name:14s} shape={tuple(tensor.shape)} dtype={tensor.dtype}")

first_40_ids = encoded_example["input_ids"][0, :40].tolist()
first_40_tokens = tokenizer.convert_ids_to_tokens(first_40_ids)
print("\nFirst 40 token IDs:", first_40_ids)
print("First 40 tokens:", first_40_tokens)

Original label: 1 -> POSITIVE
Original text preview: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and wei

Tokenizer output keys: KeysView({'input_ids': tensor([[  101,  2045,  2003,  2053,  7189,  2012,  2035,  2090,  3481,  3771,
          1998,  6337,  2099,  2021,  1996,  2755,  2008,  2119,  2024,  2610,
          2186,  2055,  6355,  6997,  1012,  6337,  2099,  3504, 15594,  2100,
          1010,  3481,  3771,  3504,  4438,  1012,  6337,  2099, 14811,  2024,
          3243,  3722,  1012,  3481,  3771,  1005,  1055,  5436,  2024,  2521,
          2062,  8552,  1012,  1012,  1012,  3481,  3771,  3504,  2062,  2066,
          3539,  8343,  1010,  2065,  2057,  2031,  2000,  3962, 12319,  10

### Checkpoint 2: Tokenization Observation

Look at the first 40 tokens printed above.

Answer in your notes:

1. Which special tokens were added by the tokenizer?
2. Did any word split into subword pieces?
3. Where do padding tokens begin, if they appear in the printed range?
4. Why does the attention mask need the same length as `input_ids`?

**Example answer style:** `input_ids` stores vocabulary IDs, while `attention_mask` tells the model which positions are real text and which positions are padding.


## 12. Tokenize the Dataset

The preprocessing function receives a batch like this:

```python
batch["text"] -> list[str]
```

and returns tokenized fields like this:

```python
{
    "input_ids": ...,
    "attention_mask": ...
}
```

**What:** apply the tokenizer to every review in the project subset.

**Why:** the Trainer expects numeric model inputs, not raw text strings.

**Example:** we remove the original `text` column from the tokenized training dataset because the model does not use it directly.

**Note:** the original text still exists in `small_datasets`, which we can use later for error analysis.


In [11]:
def tokenize_batch(batch):
    # `batch["text"]` is a list of review strings because we call `.map(..., batched=True)`.
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


# Tokenize the full project dataset once so later cells can focus on modeling rather than raw strings.
tokenized_datasets = small_datasets.map(
    tokenize_batch,
    batched=True,
    # After tokenization, the model no longer needs the original raw text column.
    remove_columns=["text"],
)

print(tokenized_datasets)
print("Tokenized columns:", tokenized_datasets["train"].column_names)
print("One tokenized row:", tokenized_datasets["train"][0])

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1200
    })
    eval: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 300
    })
})
Tokenized columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
One tokenized row: {'label': 1, 'input_ids': [101, 2045, 2003, 2053, 7189, 2012, 2035, 2090, 3481, 3771, 1998, 6337, 2099, 2021, 1996, 2755, 2008, 2119, 2024, 2610, 2186, 2055, 6355, 6997, 1012, 6337, 2099, 3504, 15594, 2100, 1010, 3481, 3771, 3504, 4438, 1012, 6337, 2099, 14811, 2024, 3243, 3722, 1012, 3481, 3771, 1005, 1055, 5436, 2024, 2521, 2062, 8552, 1012, 1012, 1012, 3481, 3771, 3504, 2062, 2066, 3539, 8343, 1010, 2065, 2057, 2031, 2000, 3962, 12319, 1012, 1012, 1012, 1996, 2364, 2839, 2003, 5410, 1998, 6881, 2080, 1010, 2021, 2031, 1000, 17936, 6767, 7054, 3401, 1000, 1012, 2111, 2066, 2000, 12826, 1010, 2000, 3648, 1010, 2000, 16157

## 13. Dynamic Padding with a Data Collator

Reviews have different lengths.

**What:** pad each batch only up to the longest review inside that batch.

**Why:** this saves compute compared with padding every review in the whole dataset to `MAX_LENGTH`.

**Example:** if a batch has sequence lengths `[58, 91, 120, 75]`, dynamic padding pads to `120`, not `256`.

**Note:** on CUDA GPUs, padding to a multiple of 8 can improve throughput.


In [12]:
# The collator pads each batch on the fly to the longest example in that batch.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    # Padding to a multiple of 8 can improve throughput on many CUDA tensor cores.
    pad_to_multiple_of=8 if device.type == "cuda" else None,
)

# Pull a few examples to preview exactly what the collator creates.
sample_features = [tokenized_datasets["train"][i] for i in range(4)]
batch = data_collator(sample_features)

print("Batch keys:", batch.keys())
for name, tensor in batch.items():
    print(f"{name:14s} shape={tuple(tensor.shape)} dtype={tensor.dtype}")

Batch keys: KeysView({'input_ids': tensor([[  101,  2045,  2003,  2053,  7189,  2012,  2035,  2090,  3481,  3771,
          1998,  6337,  2099,  2021,  1996,  2755,  2008,  2119,  2024,  2610,
          2186,  2055,  6355,  6997,  1012,  6337,  2099,  3504, 15594,  2100,
          1010,  3481,  3771,  3504,  4438,  1012,  6337,  2099, 14811,  2024,
          3243,  3722,  1012,  3481,  3771,  1005,  1055,  5436,  2024,  2521,
          2062,  8552,  1012,  1012,  1012,  3481,  3771,  3504,  2062,  2066,
          3539,  8343,  1010,  2065,  2057,  2031,  2000,  3962, 12319,  1012,
          1012,  1012,  1996,  2364,  2839,  2003,  5410,  1998,  6881,  2080,
          1010,  2021,  2031,  1000, 17936,  6767,  7054,  3401,  1000,  1012,
          2111,  2066,  2000, 12826,  1010,  2000,  3648,  1010,  2000, 16157,
          1012,  2129,  2055,  2074,  9107,  1029,  6057,  2518,  2205,  1010,
          2111,  3015,  3481,  3771,  3504,  2137,  2021,  1010,  2006,  1996,
          2060,  

### Mini Exercise: Dynamic Padding

Use the batch shapes printed above to reason about padding.

1. Is the batch length always equal to `MAX_LENGTH`?
2. Why can dynamic padding reduce wasted computation?
3. What might happen if we forgot to pad examples in a batch?

**Note:** this is a small but important production habit. Efficient batching matters a lot when datasets become large.


## 14. Load the Model

`AutoModelForSequenceClassification` does two jobs:

1. Loads the pretrained encoder backbone from `MODEL_CHECKPOINT`.
2. Adds a new two-class classification head for IMDB sentiment.

**What:** create a sentiment classifier from a pretrained language encoder.

**Why:** the encoder already understands many language patterns; fine-tuning teaches it the specific task.

**Example flow:**

```text
input_ids [B, L]
attention_mask [B, L]
-> embeddings [B, L, D]
-> transformer encoder layers [B, L, D]
-> sequence summary [B, D]
-> classifier head [B, 2]
```

**Note:** the classifier head starts randomly initialized, so predictions before training are not meaningful.


In [13]:
# The config stores label metadata and any task-specific settings before model construction.
config = AutoConfig.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

# This loads pretrained encoder weights and attaches a two-class classifier head.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    config=config,
)
# Move the whole model to the chosen accelerator before the first forward pass.
model.to(device)

print("Model class:", type(model).__name__)
print("Number of labels:", model.config.num_labels)
print("Hidden size:", model.config.dim if hasattr(model.config, "dim") else model.config.hidden_size)
print("Label mapping:", model.config.id2label)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model class: DistilBertForSequenceClassification
Number of labels: 2
Hidden size: 768
Label mapping: {0: 'NEGATIVE', 1: 'POSITIVE'}


## 15. Forward Pass Shape Check

Before training, always pass one batch through the model.

**What:** check that the tokenizer output, labels, device, and model agree.

**Why:** shape checks catch problems early, before a long training run fails.

**Example:** logits should have shape `[batch_size, 2]` because this is a two-label classification problem.

**Note:** when `labels` are included, Hugging Face sequence classifiers automatically compute cross-entropy loss.


In [14]:
# Evaluation mode disables dropout so the shape check is deterministic.
model.eval()
# Every tensor in the batch must be moved to the same device as the model.
batch_on_device = {name: tensor.to(device) for name, tensor in batch.items()}

# We do not need gradients for a simple inspection pass.
with torch.no_grad():
    # `**batch_on_device` unpacks the dictionary into named model arguments.
    outputs = model(**batch_on_device)

print("Output object:", type(outputs).__name__)
print("Loss:", outputs.loss.item())
print("Logits shape:", tuple(outputs.logits.shape))
print("Expected logits shape: [batch_size, num_labels] ->", (batch["input_ids"].shape[0], 2))

# Softmax converts raw logits into class probabilities that sum to 1.
probabilities = torch.softmax(outputs.logits, dim=-1)
print("Probabilities shape:", tuple(probabilities.shape))
print("First example probabilities:", probabilities[0].detach().cpu().numpy())

Output object: SequenceClassifierOutput
Loss: 0.680947482585907
Logits shape: (4, 2)
Expected logits shape: [batch_size, num_labels] -> (4, 2)
Probabilities shape: (4, 2)
First example probabilities: [0.48980632 0.5101937 ]


### Checkpoint 3: Shape Reasoning

Before training, explain these shapes in your own words:

1. `input_ids`: why is it `[batch_size, sequence_length]`?
2. `labels`: why is it `[batch_size]`?
3. `logits`: why is it `[batch_size, 2]`?
4. `probabilities`: why does each row sum to 1?

**Why this matters:** if you understand shapes, debugging transformer code becomes much less mysterious.


## 16. Metric Function

The `Trainer` calls this function during evaluation.

**What:** convert logits into predicted class IDs and compute accuracy, precision, recall, and F1.

**Why:** one metric rarely tells the whole story, especially when classes are imbalanced.

**Example:** high recall with low precision means the model finds many positives but also calls too many negatives positive.

**Note:** for this balanced binary dataset, accuracy is useful, but F1 is still a good habit.


In [15]:
def compute_metrics(eval_pred):
    # Trainer passes a tuple: raw model predictions first, true labels second.
    logits, labels = eval_pred
    # The highest-logit class becomes the predicted sentiment label.
    predictions = np.argmax(logits, axis=-1)

    # Convert to plain integer arrays to avoid dtype surprises in boolean comparisons.
    labels = labels.astype(int)
    predictions = predictions.astype(int)

    # Accuracy is simply the fraction of exact label matches.
    accuracy = float((predictions == labels).mean())

    # These counts are the ingredients for precision, recall, and F1.
    true_positive = int(((predictions == 1) & (labels == 1)).sum())
    false_positive = int(((predictions == 1) & (labels == 0)).sum())
    false_negative = int(((predictions == 0) & (labels == 1)).sum())

    # `max(..., 1)` avoids division by zero on tiny or degenerate batches.
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

## 17. Version-Safe TrainingArguments

Transformers changes small argument names across releases.

**What:** build `TrainingArguments` while keeping only arguments supported by the installed version.

**Why:** this makes the notebook more stable across Colab, local machines, and course environments.

**Example:** some versions use `evaluation_strategy="epoch"`, while newer versions may use `eval_strategy="epoch"`.

**Note:** `report_to="none"` disables external experiment trackers so the notebook runs quietly in local and hosted environments.


In [16]:
def make_training_args(**kwargs) -> TrainingArguments:
    # Inspect the installed signature so the notebook survives minor API renames across versions.
    signature = inspect.signature(TrainingArguments.__init__)
    params = signature.parameters

    # Older and newer Transformers releases use slightly different names for evaluation strategy.
    if "evaluation_strategy" in kwargs and "evaluation_strategy" not in params and "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    if "eval_strategy" in kwargs and "eval_strategy" not in params and "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")

    # Keep only the arguments supported by the currently installed version.
    supported_kwargs = {key: value for key, value in kwargs.items() if key in params}
    dropped = sorted(set(kwargs) - set(supported_kwargs))
    if dropped:
        print("Dropped unsupported TrainingArguments:", dropped)

    return TrainingArguments(**supported_kwargs)


# Centralized training arguments make it easy to discuss how each hyperparameter changes behavior.
training_args = make_training_args(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none",
    fp16=(device.type == "cuda"),
)

training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval

In [17]:
def build_trainer() -> Trainer:
    # Trainer also changes a bit across versions, so we inspect its accepted constructor fields.
    trainer_signature = inspect.signature(Trainer.__init__)
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["eval"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # Newer releases prefer `processing_class`; older ones still use `tokenizer`.
    if "processing_class" in trainer_signature.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    return Trainer(**trainer_kwargs)


# Instantiate once so later cells can train, evaluate, and predict using the same object.
trainer = build_trainer()
trainer

## 18. Train

Fine-tuning means starting from pretrained language knowledge and adjusting weights for IMDB sentiment.

**What:** run the Hugging Face Trainer training loop.

**Why:** training updates the encoder and classifier head so logits become useful for this task.

**Example:** with one epoch on a project subset, results should be good enough to study the workflow, not necessarily leaderboard quality.

**Note:** if training is too slow, lower `MAX_TRAIN_EXAMPLES` or set `RUN_TRAINING = False` while learning the code path.


In [18]:
if RUN_TRAINING:
    # This launches the full fine-tuning loop managed by Hugging Face Trainer.
    train_result = trainer.train()
    print("Training metrics:")
    print(train_result.metrics)
else:
    print("RUN_TRAINING=False, so training was skipped.")

/Users/suvom/anaconda3/envs/ai_ecomm/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.414399,0.436985,0.816667,0.832168,0.793333,0.812287


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training metrics:
{'train_runtime': 44.8059, 'train_samples_per_second': 26.782, 'train_steps_per_second': 3.348, 'total_flos': 79480439193600.0, 'train_loss': 0.5661749903361003, 'epoch': 1.0}


## 19. Evaluate

Evaluation uses examples the model did not train on.

**What:** compute metrics on the held-out evaluation subset.

**Why:** training performance alone can hide memorization or overfitting.

**Example:** `eval_accuracy = 0.88` means 88% of evaluation reviews were classified correctly.

**Note:** if `RUN_TRAINING = False`, metrics describe the mostly random classifier head and should not be treated as model quality.


In [19]:
# `evaluate()` runs the eval split and returns a metrics dictionary.
eval_metrics = trainer.evaluate()
eval_metrics

/Users/suvom/anaconda3/envs/ai_ecomm/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.414399,0.436985,1,0.816667,0.832168,0.793333,0.812287


{'eval_loss': 0.4369845688343048,
 'eval_accuracy': 0.8166666666666667,
 'eval_precision': 0.8321678321678322,
 'eval_recall': 0.7933333333333333,
 'eval_f1': 0.8122866894197952}

### Checkpoint 4: Metric Interpretation

After evaluation, answer:

1. Which metric is highest?
2. Which metric is lowest?
3. Does the model seem balanced between positive and negative predictions?
4. Are the results meaningful, or did you skip training?

**Note:** metrics are measurements, not decorations. Always connect them back to model behavior.


## 20. Inspect Predictions

Summary metrics are helpful, but they do not show individual mistakes.

**What:** collect raw logits, true labels, predicted labels, and confusion counts.

**Why:** error inspection helps you understand what the model is learning and where it struggles.

**Example:** a review with both praise and criticism may be harder than a review with obvious sentiment words.

**Note:** this section gives you evidence for the final project report.


In [20]:
# `predict()` gives us raw logits so we can inspect errors beyond summary metrics.
pred_output = trainer.predict(tokenized_datasets["eval"])
logits = pred_output.predictions
true_labels = pred_output.label_ids
# Convert per-class logits into a single predicted class ID.
pred_labels = np.argmax(logits, axis=-1)

print("Logits shape:", logits.shape)
print("Labels shape:", true_labels.shape)
print("Prediction label counts:", Counter(pred_labels))

# Count `(true_label, predicted_label)` pairs to build a simple confusion summary.
confusion = Counter(zip(true_labels.tolist(), pred_labels.tolist()))
print("\nConfusion matrix counts as (true, predicted):")
for key in sorted(confusion):
    print(key, "->", confusion[key])

Logits shape: (300, 2)
Labels shape: (300,)
Prediction label counts: Counter({np.int64(0): 157, np.int64(1): 143})

Confusion matrix counts as (true, predicted):
(0, 0) -> 126
(0, 1) -> 24
(1, 0) -> 31
(1, 1) -> 119


### Checkpoint 5: Prediction Inspection

Use the confusion counts above to answer:

1. How many positive reviews were predicted as negative?
2. How many negative reviews were predicted as positive?
3. Which mistake type would matter more in a real review moderation system?

**Why this matters:** two models can have similar accuracy but very different error patterns.


## 21. Inference on New Reviews

The pipeline wraps the full prediction flow.

```text
text -> tokenizer -> model -> softmax -> label
```

**What:** test the fine-tuned classifier on new review snippets.

**Why:** inference is how a trained model is used after the training notebook is finished.

**Example:** write one clearly positive review, one clearly negative review, and one mixed review.

**Note:** use inference only after training if you want meaningful predictions.


In [21]:
# The pipeline API is the quickest way to demo inference after fine-tuning.
if device.type == "cuda":
    # Pipelines expect CUDA devices as integer IDs.
    inference_device = 0
else:
    # Move back to CPU for environments where the pipeline cannot use the current accelerator directly.
    trainer.model.to("cpu")
    inference_device = -1

sentiment_pipeline = pipeline(
    task="text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=MAX_LENGTH,
    device=inference_device,
)

# These fresh examples were not part of the training subset.
new_reviews = [
    "The movie had a slow start, but the final act was powerful and emotional.",
    "I could not care about any character. The dialogue was awful.",
    "The visuals were beautiful, although the story was a little predictable.",
]

for review, prediction in zip(new_reviews, sentiment_pipeline(new_reviews)):
    print("\nReview:", review)
    print("Prediction:", prediction)


Review: The movie had a slow start, but the final act was powerful and emotional.
Prediction: {'label': 'POSITIVE', 'score': 0.6679666638374329}

Review: I could not care about any character. The dialogue was awful.
Prediction: {'label': 'NEGATIVE', 'score': 0.7398696541786194}

Review: The visuals were beautiful, although the story was a little predictable.
Prediction: {'label': 'POSITIVE', 'score': 0.5975956320762634}


## 22. Save the Fine-Tuned Model

Saving writes the files needed to reload the project later.

**What:** save model weights, config, label mappings, and tokenizer files.

**Why:** a trained model is only useful if you can load it again without rerunning training.

**Example:** later you can use `AutoTokenizer.from_pretrained(SAVE_DIR)` and `AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)`.

**Note:** saving is skipped when `RUN_TRAINING = False` because there is no fine-tuned checkpoint to preserve.


In [22]:
SAVE_DIR = OUTPUT_DIR / "final_model"

if RUN_TRAINING:
    # Save model weights/config and tokenizer files so the fine-tuned checkpoint can be reloaded later.
    trainer.save_model(str(SAVE_DIR))
    tokenizer.save_pretrained(str(SAVE_DIR))
    print("Saved to:", SAVE_DIR)
else:
    print("Training was skipped, so model saving was skipped.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: outputs/bert_imdb_sentiment_project_notebook/final_model


## 23. Full Data Flow Recap

Use this as your board summary before writing the project report.

| Stage | Tensor / Object | Shape |
|---|---|---:|
| Raw text | review string | Python `str` |
| Token IDs | `input_ids` | `[B, L]` |
| Attention mask | `attention_mask` | `[B, L]` |
| Labels | `labels` | `[B]` |
| Embeddings | internal | `[B, L, D]` |
| Encoder output | internal | `[B, L, D]` |
| Classification representation | internal | `[B, D]` |
| Classifier logits | `logits` | `[B, 2]` |
| Probabilities | `softmax(logits)` | `[B, 2]` |

**What:** the encoder reads the whole review at once and predicts one label for the whole sequence.

**Why:** sentiment classification needs a sentence/review-level decision, not one prediction per token.

**Example:** a long review still becomes one final positive/negative prediction.

**Note:** `L` is the sequence length after tokenization and padding. It can vary by batch when dynamic padding is used.


## 24. Final Project Tasks

Complete these tasks after the notebook runs end to end.

1. **Baseline run:** record `MAX_LENGTH`, `MAX_TRAIN_EXAMPLES`, `NUM_EPOCHS`, and the final evaluation metrics.
2. **Tokenization explanation:** choose one review and explain what `input_ids` and `attention_mask` mean for that example.
3. **Shape explanation:** explain why logits have shape `[batch_size, 2]`.
4. **Experiment:** change one setting, such as `MAX_LENGTH` or `MAX_TRAIN_EXAMPLES`, then compare runtime and metrics.
5. **Error analysis:** inspect at least three wrong predictions and describe what may have confused the model.
6. **Inference demo:** write three new reviews and report the model predictions.

### Report template

```text
Project title:
Model checkpoint:
Dataset:
Baseline settings:
Baseline metrics:
Experiment changed:
Experiment metrics:
Most interesting mistake:
What I learned about tokenization:
What I learned about encoder models:
One improvement I would try next:
```

**Note:** your report should use evidence from notebook outputs, not only opinions.


## 25. Optional Helper: Most Confident Wrong Predictions

This helper is useful for the final error-analysis task.

**What:** find wrong predictions where the model was highly confident.

**Why:** confident mistakes often reveal the most interesting model weaknesses.

**Example:** a sarcastic review may contain positive words but express a negative opinion.

**Note:** this cell needs `logits`, `true_labels`, and `pred_labels`, so run the prediction section first.


In [23]:
def show_top_confident_mistakes(text_dataset, logits, true_labels, pred_labels, top_k=5):
    """Print the most confident wrong predictions for project error analysis."""
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    confidence = probabilities.max(axis=-1)
    wrong_indices = np.where(pred_labels != true_labels)[0]

    if len(wrong_indices) == 0:
        print("No wrong predictions found in this evaluation subset.")
        return

    ranked_wrong = sorted(wrong_indices, key=lambda idx: confidence[idx], reverse=True)[:top_k]

    for rank, idx in enumerate(ranked_wrong, start=1):
        true_id = int(true_labels[idx])
        pred_id = int(pred_labels[idx])
        review = text_dataset[int(idx)]["text"].replace(chr(10), " ")
        print(f"\nMistake {rank}")
        print("True label:     ", id2label[true_id])
        print("Predicted label:", id2label[pred_id])
        print("Confidence:     ", round(float(confidence[idx]), 4))
        print("Review preview: ", review[:500])


if "logits" in globals() and "pred_labels" in globals():
    show_top_confident_mistakes(small_datasets["eval"], logits, true_labels, pred_labels, top_k=5)
else:
    print("Run the prediction inspection section before this helper.")



Mistake 1
True label:      POSITIVE
Predicted label: NEGATIVE
Confidence:      0.8651
Review preview:  i was having a horrid day but this movie grabbed me, and i couldn't put it down until the end... and i had forgotten about my horrid day. and the ending... by the way... where is the sequel!!!<br /><br />the budget is obviously extremely low... but ... look what they did with it! it reminds me of a play... they are basically working with a tent, a 'escape pod', a few guns, uniforms, camping gear, and a 'scanner' thing. that is it for props. Maybe this is even a good thing, forcing the acting and

Mistake 2
True label:      POSITIVE
Predicted label: NEGATIVE
Confidence:      0.852
Review preview:  I don't understand why this movie has such a low rating. It totally deserves more! Sure, it's completely ridiculous, but that's what it was supposed to be. Don't expect cinematic transcendence from a movie about beauty pageant contestants stranded on a Caribbean island! What you should expec